<a href="https://colab.research.google.com/github/durgalipu17-oss/House-Price-Prediction/blob/main/HousePrice.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline
from sklearn.preprocessing import OneHotEncoder, MinMaxScaler
from keras.models import Sequential
from keras.layers import Flatten, InputLayer, Dense
import keras

In [ ]:
train=pd.read_csv('/content/trainHousePrice.csv')

In [ ]:
trainX,trainY=train.iloc[:,:train.shape[1]-1],train.iloc[:,train.shape[1]-1]

In [ ]:
categoricals=trainX.loc[:,trainX.dtypes=='O'].columns
print(categoricals)

Index(['MSZoning', 'Street', 'Alley', 'LotShape', 'LandContour', 'Utilities',
       'LotConfig', 'LandSlope', 'Neighborhood', 'Condition1', 'Condition2',
       'BldgType', 'HouseStyle', 'RoofStyle', 'RoofMatl', 'Exterior1st',
       'Exterior2nd', 'MasVnrType', 'ExterQual', 'ExterCond', 'Foundation',
       'BsmtQual', 'BsmtCond', 'BsmtExposure', 'BsmtFinType1', 'BsmtFinType2',
       'Heating', 'HeatingQC', 'CentralAir', 'Electrical', 'KitchenQual',
       'Functional', 'FireplaceQu', 'GarageType', 'GarageFinish', 'GarageQual',
       'GarageCond', 'PavedDrive', 'PoolQC', 'Fence', 'MiscFeature',
       'SaleType', 'SaleCondition'],
      dtype='object')


In [ ]:
cat_features=trainX.loc[:,categoricals]
cat_features=cat_features.fillna(cat_features.mode().iloc[0,:])

In [ ]:
ohe=OneHotEncoder(handle_unknown='ignore')
res=ohe.fit_transform(cat_features).toarray()
print(res.shape)

(1460, 251)


In [ ]:
cols=np.array([])
for i in range(cat_features.shape[1]):
  cols=np.concatenate((cols,categoricals[i]+'_'+ np.sort(cat_features.iloc[:,i].unique())))
cat=pd.DataFrame(res,columns=cols)
print(cat.shape)

(1460, 251)


In [ ]:
trainX=trainX.drop(categoricals,axis=1)


In [ ]:
trainX=pd.concat([trainX,cat],axis=1)
print(trainX.shape)

(1460, 288)


In [ ]:
trainX.fillna(trainX.median(),inplace=True)

In [ ]:
scalar = MinMaxScaler()
norm_train = pd.DataFrame(scalar.fit_transform(trainX), columns=trainX.columns)

In [ ]:
scalar_target = MinMaxScaler()
trainY = scalar_target.fit_transform(trainY.values.reshape(-1, 1))

In [ ]:
from keras.layers import Input
model = Sequential([
  Input( shape = (norm_train.shape[1],)),
  Dense(units=256, activation='relu'),
  Dense(units=128, activation='relu'),
  Dense(units=64, activation='relu'),
  Dense(units=32, activation='relu'),
  Dense(units=16, activation='relu'),
  Dense(units=1, activation='linear')
])

In [ ]:
model.summary()

Model: "sequential_5"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_20 (Dense)                │ (None, 256)            │        73,984 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_21 (Dense)                │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_22 (Dense)                │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_23 (Dense)                │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_24 (Dense)                │ (None, 16)             │           528 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_25 (Dense)                │ (None, 1)              │            17 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 117,761 (460.00 KB)

 Trainable params: 117,761 (460.00 KB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
model.compile(optimizer='adam', loss='mean_squared_error')
model.fit(norm_train,
          trainY,
          batch_size=512,
          epochs=40,
          verbose=1,
          validation_split=0.2
          )

Epoch 1/40
3/3 ━━━━━━━━━━━━━━━━━━━━ 3s 123ms/step - loss: 0.0394 - val_loss: 0.0119
Epoch 2/40
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 42ms/step - loss: 0.0094 - val_loss: 0.0076
Epoch 3/40
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 49ms/step - loss: 0.0058 - val_loss: 0.0061
Epoch 4/40
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 44ms/step - loss: 0.0045 - val_loss: 0.0048
Epoch 5/40
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 42ms/step - loss: 0.0036 - val_loss: 0.0048
Epoch 6/40
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 40ms/step - loss: 0.0033 - val_loss: 0.0038
Epoch 7/40
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 40ms/step - loss: 0.0027 - val_loss: 0.0035
Epoch 8/40
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 40ms/step - loss: 0.0024 - val_loss: 0.0031
Epoch 9/40
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 40ms/step - loss: 0.0020 - val_loss: 0.0032
Epoch 10/40
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 40ms/step - loss: 0.0018 - val_loss: 0.0028
Epoch 11/40
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 40ms/step - loss: 0.0016 - val_loss: 0.0027
Epoch 12/40
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 51ms/step - loss: 0.0015 - val_loss: 0.0027
